# Machine-learning and structured neural pricing models under heavy-tailed losses — full pipeline

Run the cells in order from the repository root. Every long cell is resumable: after an interruption, restart the
kernel, run the setup cell and re-run the interrupted cell; finished fits are read from disk and skipped.

Data: `data/freMTPL2freq.csv` and `data/freMTPL2sev.csv` (see README). All paths are relative to the repository root.

| section | what |
|---|---|
| 1 | smoke test of the tuning and main-run code |
| 2–5 | tuning (EBM, GBM, networks, confirmation) → `tuning/selected_configs.json` |
| 6 | review of the selected configurations |
| 7 | main run, 50 splits (seeds 41–90) |
| 8 | analysis R1–R11 |
| 9 | Neural GA2M+ (fixed cap 0.995, mean-rank rule) |
| 10 | stored predictions and the calibration study |
| 11 | proposed calibrator: groupings and credibility constant k |
| 12 | figures and tables of the paper |
| 13 | appendix tables and figures |
| 14 | clipped versus unclipped cells |
| 15 | grouping chosen on the validation set |
| 16 | paired-comparison tables with bootstrap confidence intervals |

In [ ]:
import os, sys, json, subprocess
import pandas as pd

sys.path.insert(0, os.path.abspath("src"))

DATA_DIR = "data"
TUNE_DIR, RUN_DIR = "tuning", "v2_full"
CONFIGS = f"{TUNE_DIR}/selected_configs.json"
SCREEN_DIR = "v2_ga2mplus_screen995"
RULE = "meanrank"                                  # pair-selection rule of Neural GA2M+ (see section 9.3)
FIX_DIR = f"v2_ga2mplus_c995_{RULE}"
MERGED_DIR = "v2_full_plus995"
PRED_DIR, CAL_DIR, AGE_DIR, OUT_DIR = "v2_predictions", "v2_calibration", "v2_age_calibration", "v2_paper"
CLIP_DIR, SEL_DIR = "v2_clipping", "v2_calibrator_selection"
PRED_DIRS = [PRED_DIR, f"{FIX_DIR}/predictions"]
N_JOBS = max(1, (os.cpu_count() or 2) // 2)       # parallel FAST calls; results do not depend on it

import paper2_models as pm
import paper2_tuning as tu
import paper2_run_full as rf
import paper2_analysis_full as pa
import ga2m_plus as gp
import paper2_predictions as pp
import paper2_calibration_study as cs
import paper2_age_calibration as ac
import paper2_paper_figures as pf
import paper2_appendix as rx
import paper2_clipping as clp
import paper2_calibrator_selection as sel
rf.check_dependencies()

df_freq = pd.read_csv(f"{DATA_DIR}/freMTPL2freq.csv")
df_sev = pd.read_csv(f"{DATA_DIR}/freMTPL2sev.csv")
print(f"data: {len(df_freq):,} policies, {len(df_sev):,} claims")

## 1. Smoke test — must end with `SMOKE TEST PASSED`

In [ ]:
checks = tu.smoke_test(df_freq, df_sev)

## 2. Tuning stage E — EBM (10 seeds)
The table at the end shows whether the round cap was binding (`share_bags_at_cap`).

In [ ]:
tu.run_tuning(df_freq, df_sev, out_dir=TUNE_DIR, stages=("E",))

## 3. Tuning stage G — GBM (num_leaves × min_child_samples × categorical treatment, 5 seeds)

In [ ]:
tu.run_tuning(df_freq, df_sev, out_dir=TUNE_DIR, stages=("G",))

## 4. Tuning stages 1, 1b, 2, 2c — LocalGLMnet, Tweedie-CANN, Neural GA2M (5 seeds)
1 = learning rate × stopping criterion (× scheduler for GA2M); 1b = gradient clipping; 2 = capacity;
2c = GA2M bagging (5 networks only if better by more than one SE). The stages can be run one at a time, e.g. `stages=("1",)`.

In [ ]:
tu.run_tuning(df_freq, df_sev, out_dir=TUNE_DIR, stages=("1", "1b", "2", "2c"))

## 5. Tuning stage 3 — confirmation on 10 seeds; writes `tuning/selected_configs.json`

In [ ]:
tu.run_tuning(df_freq, df_sev, out_dir=TUNE_DIR, stages=("3",))

## 6. Review of the selected configurations
Selected configuration per model, its validation-deviance difference to the reference configuration (also in SE units)
and the minutes per fit. The main run uses exactly this file.

In [ ]:
sel_cfg = json.load(open(CONFIGS))
res = pd.read_csv(f"{TUNE_DIR}/results.csv")
refs = {"EBM": tu.EBM_REF, "GBM": tu.GBM_REF, **tu.NN_REF}
rows = []
for m in rf.CONFIGURED:
    cid, rid = tu.config_id(sel_cfg[m]), tu.config_id(refs[m])
    d = res[(res.model == m) & res.seed.isin(tu.SEEDS_10)].pivot_table(index="seed", columns="config_id",
                                                                      values="valid_dev_level")
    diff = (d[cid] - d[rid]).dropna() if (cid in d and rid in d) else pd.Series(dtype=float)
    se = diff.std(ddof=1) / len(diff) ** 0.5 if len(diff) > 1 else float("nan")
    rows.append({"model": m, "selected": tu.describe(m, sel_cfg[m]), "changed vs reference": cid != rid,
                 "diff vs reference (valid dev, level)": diff.mean() if len(diff) else 0.0,
                 "in SE units": diff.mean() / se if (len(diff) > 1 and se > 0) else float("nan"),
                 "seeds": len(diff),
                 "minutes per fit": res[(res.model == m) & (res.config_id == cid)].minutes.mean()})
review = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 120):
    display(review)

## 7. Main run — 50 splits, all models
Resumable. A run started with one configuration file refuses to continue with a different one.

In [ ]:
perf = rf.run_full(df_freq, df_sev, seeds=range(41, 91), out_dir=RUN_DIR, configs_path=CONFIGS)
print(perf.groupby("method").seed.nunique())

## 8. Analysis R1–R11
Tables (CSV + LaTeX) and figures (PDF) are written to `v2_full/analysis/`. The second call repeats the regular /
stressed comparisons with the model-free stress label.

In [ ]:
out = pa.run_all(RUN_DIR, group="paper")

In [ ]:
out_mf = pa.run_all(RUN_DIR, group="model_free", save=False)

## 9. Neural GA2M+
Interaction pairs screened on the training set only: target capped at the 99.5% quantile of the positive training
claim amounts, FAST ranking on 20 half-samples, the 10 pairs with the best mean rank. The Neural GA2M is then fitted
on these pairs with the configuration and seeds of the main run.

### 9.1 Smoke test (subsample) — must end with `GA2M+ SMOKE TEST PASSED`

In [ ]:
checks_plus = gp.smoke_test(df_freq, df_sev, CONFIGS, n_jobs=N_JOBS)

### 9.2 Screening at the fixed cap 0.995 (training sets only, no network; resumable)

In [ ]:
gp.screen_fixed_cap(df_freq, df_sev, seeds=range(41, 91), screen_dir=SCREEN_DIR, configs_path=CONFIGS, cap=0.995,
                    n_jobs=N_JOBS)

### 9.3 Selection rule (training sets only): stability selection (`pi`) or half-sample mean rank (`meanrank`)

In [ ]:
rr = gp.rule_report(RUN_DIR, SCREEN_DIR, cap=0.995)
print("recommended rule:", rr["recommended"], "| rule used:", RULE)

### 9.4 Neural GA2M+ on the 50 splits (resumable)

In [ ]:
perf_fix = gp.run_fixed_cap(df_freq, df_sev, seeds=range(41, 91), screen_dir=SCREEN_DIR, out_dir=FIX_DIR,
                            configs_path=CONFIGS, cap=0.995, rule=RULE)

### 9.5 Comparison, structure stability, merge with the main run and R1–R11 with Neural GA2M+

In [ ]:
gp.compare_runs(RUN_DIR, {"fixed cap 0.995": FIX_DIR})
gp.compare_runs(RUN_DIR, {"fixed cap 0.995": FIX_DIR}, ref="EBM")
stab = gp.structure_stability(RUN_DIR, FIX_DIR)
gp.merge_results(RUN_DIR, FIX_DIR, MERGED_DIR)
out_plus = pa.run_all(MERGED_DIR, group="paper")

## 10. Stored predictions and the calibration study
### 10.1 Smoke test (subsample) — must end with two `PASSED` lines

In [ ]:
checks_pred, small = pp.smoke_test(df_freq, df_sev, CONFIGS)
cs.run_study(None, None, pred_dirs=["predictions_smoke"], out_dir="calibration_smoke", seeds=[49], n_perm=9,
             df_prepared=small)
r_s, h_s = cs.load("calibration_smoke")
ok = (r_s.grouping.nunique() == len(cs.GROUPINGS)
      and r_s[["deviance", "gini", "seg_misalloc"]].notna().all().all())
print("CALIBRATION STUDY SMOKE TEST PASSED" if ok else "CALIBRATION STUDY SMOKE TEST FAILED")

### 10.2 Raw predictions of the seven main-run models (refit with the main-run configurations; resumable)

In [ ]:
ver = pp.archive_predictions(df_freq, df_sev, seeds=range(41, 91), out_dir=PRED_DIR, main_dir=RUN_DIR,
                             configs_path=CONFIGS)

### 10.3 Calibration study (resumable) and consistency check with the main run

In [ ]:
r_cal, h_cal = cs.run_study(df_freq, df_sev, pred_dirs=PRED_DIRS, out_dir=CAL_DIR, seeds=range(41, 91))
cons = cs.consistency_check(CAL_DIR, RUN_DIR)

In [ ]:
out_cal = cs.report(CAL_DIR)

## 11. Proposed calibrator: clipped credibility cells after global rescaling
Cell groupings × credibility constant k, from the stored predictions (no refit; resumable).

In [ ]:
var = ac.compute_variants(df_freq, df_sev, pred_dirs=PRED_DIRS, out_dir=AGE_DIR, seeds=range(41, 91))

In [ ]:
age_tabs = ac.report(CAL_DIR, AGE_DIR)
diag, cells = ac.cell_diagnostics(df_freq, df_sev, PRED_DIRS, out_dir=AGE_DIR)

## 12. Figures and tables of the paper
### 12.1 Policy-level pass over the 50 splits (stored predictions only)

In [ ]:
pol = pf.policy_pass(df_freq, df_sev, PRED_DIRS, OUT_DIR)

### 12.2 Neural GA2M+: pair stability, deviance decomposition, extreme-claim policies, balance and Gini

In [ ]:
tg = pf.ga2m_report(RUN_DIR, FIX_DIR, CAL_DIR, OUT_DIR)

### 12.3 Calibration methods

In [ ]:
tc = pf.calibration_report(CAL_DIR, AGE_DIR, OUT_DIR)

## 13. Appendix tables and figures

In [ ]:
a_bal = rx.signed_balance_table(df_freq, df_sev, PRED_DIRS, OUT_DIR)
a_stress = rx.model_free_stress(MERGED_DIR, OUT_DIR)
a_rel = rx.relativity_figure(MERGED_DIR, OUT_DIR, df_freq)
a_pd = rx.pd_figure(MERGED_DIR, OUT_DIR)
rx.check_numbers(df_freq, df_sev, RUN_DIR, FIX_DIR)

## 14. Clipped versus unclipped credibility cells, and the clipping bounds

In [ ]:
res_clip = clp.run(df_freq, df_sev, PRED_DIRS, out_dir=CLIP_DIR)
tabs_clip = clp.summary(res_clip, out_dir=CLIP_DIR)
clp.tradeoff_figure(res_clip, out_dir=CLIP_DIR, paper_dir=OUT_DIR)
clp.bounds_figure(res_clip, out_dir=CLIP_DIR)

## 15. Cell grouping chosen on the validation set by cross-fitting

In [ ]:
res_sel = sel.run(df_freq, df_sev, PRED_DIRS, out_dir=SEL_DIR)
tabs_sel = sel.summary(res_sel, out_dir=SEL_DIR)
sel.figure(res_sel, out_dir=SEL_DIR)

## 16. Paired comparisons with bootstrap confidence intervals
(a) every model except Neural GA2M+ against GBM; (b) Neural GA2M+ against Neural GA2M by claim-location group.

In [ ]:
cmd = [sys.executable, "src/paper2_extra_tables.py", "--perf", f"{MERGED_DIR}/perf.csv",
       "--dev", f"{OUT_DIR}/tables/policy_pass_deviance.csv", "--seeds", f"{OUT_DIR}/tables/policy_pass_seeds.csv",
       "--out", f"{OUT_DIR}/tables/paired"]
print(subprocess.run(cmd, check=True, capture_output=True, text=True).stdout)